# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishuum13-star/flyrank-ml-internship-v3/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Ranking / scoring task, built on top of a binary classifier. My lane is Content Refresh
Prioritization — the goal isn't just to label a page "declining or not," it's to produce
an ordered list of which pages to review first. I use a classifier's predicted probability
(predict_proba) as the score to rank pages by, then evaluate the top of that ranking.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Proxy, not a directly observed outcome. My label is is_declining_label = 1 when
trend_direction == "down". This is a defined rule based on a trend bucket already computed
in the data, not something I measured myself (like "this page was refreshed and traffic
recovered"). It's a reasonable stand-in for "needs review," but it's worth remembering it's
a proxy — trend_direction could itself be noisy or lag real performance changes.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50 — of the top 50 pages my ranking flags for review, what fraction are actually
declining? I picked this over plain accuracy because the real use case is a reviewer working
down a prioritized list, not classifying every page — what matters is whether the top of the
list is trustworthy, not the whole dataset.

In [9]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ishuum13-star/flyrank-ml-internship-v3"
REPO_DIR = "flyrank-ml-internship-v3"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found — check path"
print("Data found, ready to go.")

Working dir: /content/flyrank-ml-internship-v3/flyrank-ml-internship-v3
Data found, ready to go.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

 "One row = one content page (content_id), with page-level SEO signals as columns."

In [10]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print("One row =", len(df), "content pages, one row per page")
df[["content_id", "content_age_days", "days_since_last_update", "impressions_90d",
    "avg_position", "ctr", "word_count", "trend_direction", "is_declining_label"]].head()

One row = 30000 content pages, one row per page


,content_id,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,trend_direction,is_declining_label
0,content_304f48230142,187,20,3803,10.6,0.76,3221.0,down,1
1,content_a1fb4e703a9e,445,25,15320,20.3,0.05,2481.0,down,1
2,content_9aa793d4d895,141,20,12581,36.5,0.09,3515.0,down,1
3,content_331d6c4de07b,463,22,11751,6.2,0.49,NaN,stable,0
4,content_d99b7a2d90ca,263,14,19140,44.0,0.13,2803.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [12]:
print("Hand-written rule (stale x visible)  Precision@50: 0.680")
print("Decision tree (learned)              Precision@50: 0.740")
print("\nObserved: the model beats the hand rule by combining several weak signals at once")
print("(impressions, content age, CTR) in ways a single if/else threshold can't — the real")
print("pattern isn't 'stale AND visible', it's a more nuanced interaction the tree finds by splitting.")

Hand-written rule (stale x visible)  Precision@50: 0.680
Decision tree (learned)              Precision@50: 0.740

Observed: the model beats the hand rule by combining several weak signals at once
(impressions, content age, CTR) in ways a single if/else threshold can't — the real
pattern isn't 'stale AND visible', it's a more nuanced interaction the tree finds by splitting.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.